# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])
print("Published Date:", metadata['datePublished'])
print("Number of Record Sets:", len(metadata.get('recordSet', [])))


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes records into `recordSet` objects. Each `recordSet` contains fields defined by their `@id`.

Let's enumerate all record sets and fields:

In [ ]:
# Find available record sets
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {getattr(rs, 'name', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', None)}, type: {getattr(field, 'data_type', None)}")
    print("  Columns:")
    for column in getattr(rs, 'columns', []):
        print(f"    - @id: {column.id}, name: {getattr(column, 'name', None)}, source: {getattr(column, 'source', None)}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference record set and field `@id`s as shown above.

Below, we extract all available record sets.

In [ ]:
# Get the list of record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rec_id in record_set_ids:
    rec_list = list(dataset.records(record_set=rec_id))
    print(f"Recordset {rec_id}: {len(rec_list)} records")
    if len(rec_list) > 0:
        df = pd.DataFrame(rec_list)
        dataframes[rec_id] = df

# Show columns for each loaded DataFrame
for rec_id, df in dataframes.items():
    print(f"Columns for record set {rec_id}: {df.columns.tolist()}")

# Print preview of first available record set
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id and first_record_set_id in dataframes:
    print(f"Preview of DataFrame for record set {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by criteria, normalizing numeric fields, categorizing/grouping records. 

Let's choose one numeric field and one group field from the available columns in the main record set.

In [ ]:
# Choose record set and fields by @id
# For demonstration: substitute these with actual @id values from your dataset
main_record_set_id = first_record_set_id

# Example column names (replace with those with numeric data and grouping, found previously)
main_df = dataframes.get(main_record_set_id, pd.DataFrame())
print("Available columns:", list(main_df.columns))
# Try to pick 'age' or similar numeric column, and 'sex' or 'MSI_Status' for grouping
numeric_field = None
group_field = None
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field = col
    if 'sex' in col.lower() or 'msi' in col.lower():
        group_field = col

if numeric_field:
    # Filter for patients older than a threshold
    threshold = 50
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of selected numeric and categorical fields from the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field and numeric_field in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Plot boxplot for numeric_field by group_field
if numeric_field and group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load tabular, clinical data using a Croissant schema and the `mlcroissant` library
- Enumerate and extract record sets and fields by their `@id`
- Perform filtering, normalization, and grouping on key data fields
- Visualize distributions and relationships in the dataset

For further analyses, consult the Croissant schema for additional metadata, and reference each field, column, or record set by its `@id` to support reproducible workflows.

Explore more at https://mlcommons.org/croissant and https://sen.science